# Demonstration of loading and visualising Geoscience Australia's Sentinel-1 IW Normalised Radar Backscatter (collection 0)

This notebook demonstrates key steps for using Python to load preliminary Sentinel-1 IW backscatter products developed by Geoscience Australia. 
The notebook covers how to load via a STAC API or the Digital Earth Australia Datacube.

More detailed loading examples for each approach can be found in notebooks 3 and 4.

## Set-up

For this notebook, you will choose upfront if you want to run the STAC approach, or the Datacube approach.
Uncomment the approach you wish to run, leaving the other approach commented.

In [ ]:
approach = "stac"
# approach = "datacube"

### Import required libraries

This imports python libraries that are used in both approaches, as well as any that are specific to the chosen approach

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from odc.geo import BoundingBox
import odc.geo.xr
import pathlib
import xarray as xr

if approach == "stac":
    from pystac_client import Client
    from odc.stac import load, configure_s3_access

if approach == "datacube": 
    import datacube

### Environment set up

In [ ]:
if approach == "stac":
    catalog = "https://explorer.dev.dea.ga.gov.au/stac"
    stac_client = Client.open(catalog)
    configure_s3_access(cloud_defaults=True, aws_unsigned=True)

if approach == "datacube": 
    dc = datacube.Datacube(env="dev", app="S1_Backscatter_Demo")

## Searching and Loading

For this demo, we have chosen Lake Sorell in Tasmania as the area of interest.

We will perform a basic search and load.
To learn more about how to customise loading, check the following notebooks:
* [03_loading_with_stac.ipynb](03_loading_with_stac.ipynb)
* [04_loading_with_datacube.ipynb](04_loading_with_datacube.ipynb)

### Set up area of interest and date range

In [ ]:
aoi_bbox = BoundingBox(
    left=147.07251,
    bottom=-42.22120,
    right=147.24274,
    top=-42.03035,
    crs="EPSG:4326"
)

start_date = "2024-06-01"
end_date = "2024-08-01"

aoi_bbox.explore()

### Set up general data loading settings

* `product_to_load`: the product, or list of products to load from.
* `measurements_to_load`: the measurements to load from the data (e.g. VV). If not specified, all measurements will be loaded.
* `output_crs`: the coordinate reference system (CRS) to project loaded data to. If not specified, the data's native CRS will be used.
* `output_res`: the resolution to load the data at, in the same units as the chosen CRS. If not specified, the data's native resolution will be used.
* `group_by`: how to group loaded data. Solar day is a sensible default.

In [ ]:
product_to_load = "ga_s1_nrb_iw_vv_vh_0"
measurements_to_load = ["VV_gamma0", "VH_gamma0", "mask"]
output_crs = "EPSG:3577"
output_res = 20 # metres
group_by = "solar_day"

### Run search and load

There is a slight difference between STAC and the Open Data Cube, in that with STAC, the finding and loading steps are treated separately, whereas for the Open Data Cube, they can happen within a single call.

Data will be lazy-loaded, meaning a summary of the data to be loaded will be provided, but nothing will be loaded into memory. This is achieved with an argument for each load command:
* for odc.stac.load, this is done with the `chunks` argument.
* for dc.load, this is done with the `dask_chunks` argument.

In [ ]:
if approach == "stac":

    stac_items = stac_client.search(
        collections=[product_to_load],
        datetime=f"{start_date}/{end_date}",
        bbox=aoi_bbox.bbox,
    ).item_collection()

    print(f"Found {len(stac_items)} items")

    ds = load(
        items=stac_items,
        bands=measurements_to_load,
        intersects=aoi_bbox.boundary(),
        crs=output_crs,
        resolution=output_res,
        groupby="solar_day",
        chunks={},
    )

if approach == "datacube":
    ds = dc.load(
        product=product_to_load,
        measurements=measurements_to_load,
        lat=(aoi_bbox.bottom, aoi_bbox.top),
        lon=(aoi_bbox.left, aoi_bbox.right),
        output_crs=output_crs,
        resolution=(output_res, -output_res),
        time=(start_date, end_date),
        group_by="solar_day",
        dask_chunks={}
    )

ds

### Loading data in memory

Once you have decided you are happy with the area of interest, crs, resolution, bands, etc., you can load data into memory using xarray's `.compute` operation. 

This is valuable once you are ready to apply transformations to the data, or wish to visualise the data, as it will save needing to read the data into memory every time.

In [ ]:
ds = ds.compute()

## Transforming and visualising loaded data

The GA Sentinel-1 backscatter products are provided as linear gamma0 values. 
It is common to apply some transformations, such as masking, speckle filtering, and converting from linear scale to decibels.

To learn more about data transformation, check the following notebook:
* [05_transforming_data.ipynb](05_transforming_data.ipynb)

### Timeseries plot
To see the full timeseries for a particular band, the following plotting command can be used:

In [ ]:
ds["VV_gamma0"].plot.imshow(col="time", col_wrap=4, robust=True, cmap="Greys_r")

### Speckle filtering

Speckle filtering aims to reduce noise present in SAR images.
One filter that is commonly applied is the Lee filter.
We have supplied a [python file](speckle_filters.py) that contains the Lee filter definition, as well as a function to apply it to xarrays.
In this example, we apply the Lee filter using a uniform filter window size of 5 pixels.


In [ ]:
from de_sar_demo.speckle_filters import apply_lee_filter

ds["VV_gamma0_filtered"] = apply_lee_filter(ds["VV_gamma0"], size=5)

#### Compare before and after speckle filtering

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 5))

ds_timestep = ds.isel(time=0)

ds_timestep["VV_gamma0"].plot(ax=axes[0], cmap="Greys_r", robust=True)
ds_timestep["VV_gamma0_filtered"].plot(ax=axes[1], cmap="Greys_r", robust=True)

plt.tight_layout()
plt.show()

### Masking

The GA Sentinel-1 backscatter product comes with a mask that indicates invalid pixels, along with pixels impacted by layover and shadow. 

The masks have the following values:
| Value | Property |
| --- | ----------- |
| 0 | Valid | 
| 1 | Shadow |
| 2 | Layover |
| 3 | Shadow and layover |
| 255 / NaN | Invalid |

The following code displays the masks and shows how to apply them.

In [ ]:
ds.mask.plot.imshow(
    col="time",
    col_wrap=4,
    levels=[-0.5, 0.5, 1.5, 2.5, 3.5], 
    cbar_kwargs={'ticks': [0, 1, 2, 3]}
)

#### Applying the mask

To apply the mask, we use xarray's where function, which takes the condition, followed by the values to keep if the condition is true, followed by the values to use if the condition is false. 
The following code creates a new band, `VV_gamma0_filtered_masked`, that keeps the original `VV_gamma0_filtered` values where the mask is equal to 0, and replaces values with NaN otherwise.

In [ ]:
ds["VV_gamma0_filtered_masked"] = xr.where(ds.mask==0, ds["VV_gamma0_filtered"], np.nan)
ds["VV_gamma0_filtered_masked"].isel(time=0).plot.imshow(cmap="Greys_r", robust=True)

### Converting to decibels

GA's Sentinel-1 backscatter product is provided as linear gamma0. 
For some analyses, it may be beneficial to convert the linear backscatter to decibels (dB).

The conversion equation is 
$${\gamma^0}_{\text{dB}} = 10 \times \log_{10}({\gamma^0}_\text{linear})$$

In [ ]:
ds["VV_gamma0_filtered_masked_db"] = 10*np.log10(ds["VV_gamma0_filtered_masked"])

ds["VV_gamma0_filtered_masked_db"].plot.imshow(col="time", col_wrap=4, cmap="Greys_r")

### Exporting

Once you have generated the data you need, you may wish to export it to enable further analysis.
The `odc-geo` python library provides a useful function for exporting cloud optimised GeoTIFFs.
First, we'll create a folder to store the outputs in.

In [ ]:
output_path = pathlib.Path("outputs")
output_path.mkdir(exist_ok=True)

#### Exporting all time steps as Cloud Optimised GeoTIFFs

In [ ]:
ds_datetime_strings = ds.time.dt.strftime("%Y-%m-%d").values

for timestep in range(len(ds.time)):
    filename = output_path / f"VV_gamma0_filtered_masked_db_{ds_datetime_strings[timestep]}.tif"
    ds["VV_gamma0_filtered_masked_db"].isel(time=timestep).odc.write_cog(filename, overwrite=True)